In [0]:
import pyspark.sql.functions as F
import pyspark.sql.types as T

In [0]:
%sql
select * from f1_racing.silver.slv_circuits

In [0]:
%sql
select * from f1_racing.silver.slv_races

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW intermediate_staged_table AS 
SELECT DISTINCT
    r.circuitId,
    c.circuit_ref,
    c.name as circuit_name,
    r.name as event_name,
    c.location,
    c.country,
    CAST(c.latitude AS STRING),
    CAST(c.longitude AS STRING),
    c.altitude,
    count(*) as total_races,
    date_format(max(r.date_time), 'dd-MM-yyyy') as recent_race_date,
    date_format(min(r.date_time), 'dd-MM-yyyy') as first_race_date
FROM f1_racing.silver.slv_races as r
LEFT JOIN  f1_racing.silver.slv_circuits as c
ON r.circuitId = c.circuit_id
GROUP BY r.circuitId,
        c.circuit_ref,
        c.name,
        c.location,
        c.country,
        c.latitude,
        c.longitude,
        c.altitude,
        r.name
ORDER BY circuitId, recent_race_date ASC

In [0]:
%sql
SHOW COLUMNS IN f1_racing.silver.slv_circuits

In [0]:
%sql
select * from f1_racing.silver.slv_results

In [0]:
%sql
SELECT * FROM intermediate_staged_table

In [0]:
df = spark.sql("""
        select 
            *
        from intermediate_staged_table
          """)

In [0]:
results_df = spark.read.table('f1_racing.silver.slv_results')
race_df = spark.read.table('f1_racing.silver.slv_races')

new_race_df = race_df.join(results_df, on= race_df['raceId'] == results_df['race_id'], how= 'left' )\
        .filter(F.col('fastest_lap_time') != -1)\
        .filter(F.col('fastest_lap_speed') != -1)\
        .select('raceId','circuitId',results_df['fastest_lap_time'].alias('time'),F.col('fastest_lap_speed').alias('speed'))
        

display(new_race_df)

In [0]:
time_speed_df = new_race_df.groupBy(F.col('circuitId')).agg(F.max('speed').alias('fastest_lap_speed_in_circuit'),F.min('time').alias('fastest_lap_time_in_circuit'))

final_df = df.join(time_speed_df, how='left' , on= 'circuitId').fillna(-1)
        

In [0]:
display(final_df)

In [0]:
final_df.write\
 .format("delta")\
 .mode("overwrite")\
 .saveAsTable("f1_racing.gold.gold_circuits")